# Qdrant

Jusqu'à présent nous avons travaillé avec numpy pour stocker les vecteurs et faire le recherche vectorielle mais cette solution ne peut pas s'étendre à un grand nombre de vecteurs.

Nous le verrons aussi plus tard mais la recherche sémantique à elle seule est rarement suffisante puisque notamment elle peut passer à côté de termes bien précis, des dates, du jargon technique, etc.

Qdrant permet de stocker un grand nombre de vecteur et de chercher dedans le meilleur candidat.

In [15]:
import numpy as np
import polars as pl

from transformers import AutoTokenizer
from docling_core.types.doc import DoclingDocument
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker

from openai import OpenAI
from qdrant_client import QdrantClient, models

import uuid

from pathlib import Path

In [16]:
CORPUS = Path('data/veltis')
EXPORT = Path('data/veltis_export')
MODELS = Path('../models')

EMBEDDING_URL = 'http://100.85.184.14:8000/v1'
EMBEDDING_MODEL = 'bge-m3'
MAX_TOKENS = 512

QDRANT_URL = 'http://localhost:6333'
QDRANT_API_KEY = '1234'
COLLECTION = 'Veltis'


qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embedder = OpenAI(base_url=EMBEDDING_URL, api_key='no-sk')

C:\Users\romainwauman\AppData\Local\Temp\ipykernel_45408\787839830.py:14: UserWarning: Api key is used with an insecure connection.
  qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


## Chunking 

In [17]:
def chunk_document(path, chunker):
    doc = DoclingDocument.load_from_json(path)
    return [
        {
            'document': path.stem,
            'chunk_index': i,
            'headings': ' | '.join(chunk.meta.headings or []),
            'text': chunker.contextualize(chunk),
            'file_extension': path.suffixes[-2].lstrip('.')
        }
        for i, chunk in enumerate(chunker.chunk(dl_doc=doc))
    ]

In [18]:
tokenizer = HuggingFaceTokenizer(tokenizer=AutoTokenizer.from_pretrained(MODELS / 'embedder' / EMBEDDING_MODEL), max_tokens=MAX_TOKENS)
chunker = HybridChunker(tokenizer=tokenizer)

chunks = []
documents = EXPORT.glob('*.json')
[chunks.extend(chunk_document(document, chunker)) for document in documents]
pl.from_dicts(chunks)

document,chunk_index,headings,text,file_extension
str,i64,str,str,str
"""grille_salariale_2026.pdf""",0,"""Grille des rémunérations et av…","""Grille des rémunérations et av…","""pdf"""
"""grille_salariale_2026.pdf""",1,"""1. Barème des salaires de base""","""1. Barème des salaires de base…","""pdf"""
"""grille_salariale_2026.pdf""",2,"""1. Barème des salaires de base""","""1. Barème des salaires de base…","""pdf"""
"""grille_salariale_2026.pdf""",3,"""2. Avantages complémentaires""","""2. Avantages complémentaires T…","""pdf"""
"""guide_onboarding.pdf""",0,"""Guide d'accueil des nouveaux c…","""Guide d'accueil des nouveaux c…","""pdf"""
…,…,…,…,…
"""rapport_annuel_2025.pdf""",7,"""6. Détail des investissements …","""6. Détail des investissements …","""pdf"""
"""rapport_annuel_2025.pdf""",8,"""6. Détail des investissements …","""6. Détail des investissements …","""pdf"""
"""rapport_annuel_2025.pdf""",9,"""6. Détail des investissements …","""6. Détail des investissements …","""pdf"""


## Embedding

In [19]:
def embed(texts, client, batch_size=32):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start+batch_size]
        r = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        vectors.extend(vector.embedding for vector in r.data)
    return vectors

np.array(embed([chunk.get('text') for chunk in chunks], embedder))


array([[-3.79097238e-02,  2.53666136e-02, -6.45287782e-02, ...,
         1.98334400e-02, -2.86753029e-02, -4.91630025e-02],
       [ 1.06750429e-02,  3.58291082e-02, -9.96602923e-02, ...,
         2.36205235e-02,  1.33139296e-02, -2.46362463e-02],
       [-2.94476468e-03,  7.29156705e-03, -7.78155252e-02, ...,
         1.06050335e-02,  1.17111392e-02,  2.15399591e-03],
       ...,
       [-9.33107827e-03, -7.49359839e-04, -4.06324752e-02, ...,
        -2.34375000e-02,  4.13053222e-02,  3.95484455e-02],
       [ 1.22931954e-02, -8.19982888e-05, -1.56831369e-02, ...,
        -2.67283861e-02,  8.80081020e-03,  3.99761694e-03],
       [-2.74326243e-02,  4.30650450e-02,  1.35882560e-03, ...,
         4.75422926e-02,  1.14492318e-02, -6.51477426e-02]],
      shape=(85, 1024))

## Générer un identifiant

Avant d'envoyer les données dans qdrant nous devons générer un identifiant stable par chunk. L'intérêt est de pouvoir éventuellement faire des upsert dans qdrant sans devoir tout réinjecter.

In [20]:
def get_uuid(chunk):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f'{chunk.get('document')} - {chunk.get('chunk_index')}'))
    
get_uuid(chunks[0])

'84f1e273-9215-537c-a0f3-ff0dde091854'

## Injection

Nous allons injecter deux vecteurs. Les denses pour la recherche sémantique et les *sparses* pour la recherche lexicale

### Création de la collection

In [21]:
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    COLLECTION,
    vectors_config={'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE)},
    sparse_vectors_config={'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)}
)

qdrant.create_payload_index(COLLECTION, field_name='document', field_schema=models.PayloadSchemaType.KEYWORD)

qdrant.collection_exists(COLLECTION)

True

### Vecteur sparse

BM25 est un algorithme destiné à faire des recherches syntaxiques et qui attribue à chaque mot une certaine podération. Cette pondération dépend de trois paramètres principalement:
1. un mot qui se répète perd en importance
2. un mot rare dans toute la collection compte beaucoup
3. un mot dans un chunk court compte plus que dans un chunk long. La notion de long et court est déterminée en fonction de la taille moyenne des chunks rencontrés.

In [22]:
def get_bm25(text):
    return models.Document(text=text, model='qdrant/bm25', options=BM25_OPTIONS)

avg_len = np.mean([len(chunk["text"].split()) for chunk in chunks])

BM25_OPTIONS = {
    'language': 'french',
    'ascii_folding': True,
    'avg_len': avg_len
    }

### Injection

In [23]:
def inject_chunk(chunks, qdrant_client, embedder_client, batch_size=128):
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        dense_vectors = embed([chunk.get('text') for chunk in batch], embedder_client)
        payload = [batch, dense_vectors]
        points = [
            models.PointStruct(
                id = get_uuid(chunk),
                vector = {'dense': dense_vector, 'sparse': get_bm25(chunk.get('text'))},
                payload = chunk
            )
            for chunk, dense_vector in zip(batch, dense_vectors)
        ]
        qdrant_client.upsert(COLLECTION, points=points)

inject_chunk(chunks, qdrant, embedder)

## La recherche

La recherche peut donc se faire de deux manières désormais. Via les vecteurs dense ou sparse

In [24]:
def search_dense(text, qdrant_client, embed_client, limit=10, query_filter=None):
    return qdrant_client.query_points(
        COLLECTION, query=embed([text], embed_client)[0], using='dense',
        limit=limit, query_filter=query_filter
    ).points

text = "Quel est la stratégie de développement de Veltis dans les prochaines années ?"
results = search_dense(text, qdrant, embedder, 3)
pl.DataFrame(results)

id,version,score,payload,vector,shard_key,order_value
str,i64,f64,struct[5],null,null,null
"""456fb1a3-02e8-515e-a12c-ab3a3b…",3,0.672905,"{""plan_strategique_2026-2028.pptx"",0,"""",""Plan stratégique « Cap 2028 » Comité de direction — janvier 2026 Où en sommes-nous ? 38,4 M€ 9 850 27 % chiffre d'affaires 2025 vélos vendus en 2025 du chiffre d'affaires à l'export Veltis SA — Comité de direction — confidentiel Trois axes stratégiques 1. Professionnels 2. Export 3. Services Devenir la référence du vélo de livraison urbaine auprès des entreprises. Ouvrir un nouveau pays par an en Europe du Nord. Entretien sur site, location longue durée et reconditionnement. Veltis SA — Comité de direction — confidentiel Trajectoire de croissance Chiffre d'affaires annuel, en millions d'euros Veltis SA — Comité de direction — confidentiel Calendrier d'ouverture des marchés export Pays-Bas, Année d'ouverture = 2026. Pays-Bas, Mode d'entrée = Distributeur exclusif. Allemagne, Année d'ouverture = 2027. Allemagne, Mode d'entrée = Filiale commerciale. Danemark, Année d'ouverture = 2028. Danemark, Mode d'entrée = Distributeur exclusif Veltis SA — Comité de direction — confidentiel Investissements clés • Deuxième site de production à Lorvan : 6,5 millions d'euros (sous réserve d'approbation du conseil d'administration) • Atelier de reconditionnement à Marnay : 0,8 million d'euros • Plateforme de suivi connecté des flottes professionnelles : 0,5 million d'euros Veltis SA — Comité de direction — confidentiel Prochaines étapes • Mars 2026 : présentation du projet de deuxième site au conseil d'administration • Avril 2026 : signature du contrat de distribution aux Pays-Bas • Juin 2026 : lancement de l'offre de location longue durée Veltis SA — Comité de direction — confidentiel"",""pptx""}",null,null,null
"""d52a0276-e516-5208-9bbc-7ba48f…",3,0.6561688,"{""rapport_annuel_2025.pdf"",11,""7. Perspectives 2026"",""7. Perspectives 2026 Pour accompagner la croissance, la direction soumettra au conseil d'administration un projet de deuxième site de production. Les priorités commerciales de 2026 sont l'ouverture du marché néerlandais et le développement d'une offre de location longue durée. À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à Valmont des vélos cargo à assistance électrique pour les familles, les professionnels et les collectivités. L'entreprise emploie 182 personnes et distribue ses vélos dans 11 pays."",""pdf""}",null,null,null
"""b35cd7a8-018a-51a4-bcad-f04c11…",3,0.638535,"{""presentation_veltis.pdf"",0,""Présentation de l'entreprise"",""Présentation de l'entreprise Veltis SA - édition 2026"",""pdf""}",null,null,null


In [25]:
def search_sparse(text, qdrant_client, limit=10, query_filter=None):
    return qdrant_client.query_points(
        COLLECTION, query=get_bm25(text), using='sparse',
        limit=limit, query_filter=query_filter
    ).points

text = "Quel est la stratégie de développement de Veltis dans les prochaines années ?"
results = search_sparse(text, qdrant, 3)
pl.DataFrame(results)

id,version,score,payload,vector,shard_key,order_value
str,i64,f64,struct[5],null,null,null
"""456fb1a3-02e8-515e-a12c-ab3a3b…",3,8.752091,"{""plan_strategique_2026-2028.pptx"",0,"""",""Plan stratégique « Cap 2028 » Comité de direction — janvier 2026 Où en sommes-nous ? 38,4 M€ 9 850 27 % chiffre d'affaires 2025 vélos vendus en 2025 du chiffre d'affaires à l'export Veltis SA — Comité de direction — confidentiel Trois axes stratégiques 1. Professionnels 2. Export 3. Services Devenir la référence du vélo de livraison urbaine auprès des entreprises. Ouvrir un nouveau pays par an en Europe du Nord. Entretien sur site, location longue durée et reconditionnement. Veltis SA — Comité de direction — confidentiel Trajectoire de croissance Chiffre d'affaires annuel, en millions d'euros Veltis SA — Comité de direction — confidentiel Calendrier d'ouverture des marchés export Pays-Bas, Année d'ouverture = 2026. Pays-Bas, Mode d'entrée = Distributeur exclusif. Allemagne, Année d'ouverture = 2027. Allemagne, Mode d'entrée = Filiale commerciale. Danemark, Année d'ouverture = 2028. Danemark, Mode d'entrée = Distributeur exclusif Veltis SA — Comité de direction — confidentiel Investissements clés • Deuxième site de production à Lorvan : 6,5 millions d'euros (sous réserve d'approbation du conseil d'administration) • Atelier de reconditionnement à Marnay : 0,8 million d'euros • Plateforme de suivi connecté des flottes professionnelles : 0,5 million d'euros Veltis SA — Comité de direction — confidentiel Prochaines étapes • Mars 2026 : présentation du projet de deuxième site au conseil d'administration • Avril 2026 : signature du contrat de distribution aux Pays-Bas • Juin 2026 : lancement de l'offre de location longue durée Veltis SA — Comité de direction — confidentiel"",""pptx""}",null,null,null
"""d52a0276-e516-5208-9bbc-7ba48f…",3,5.778693,"{""rapport_annuel_2025.pdf"",11,""7. Perspectives 2026"",""7. Perspectives 2026 Pour accompagner la croissance, la direction soumettra au conseil d'administration un projet de deuxième site de production. Les priorités commerciales de 2026 sont l'ouverture du marché néerlandais et le développement d'une offre de location longue durée. À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à Valmont des vélos cargo à assistance électrique pour les familles, les professionnels et les collectivités. L'entreprise emploie 182 personnes et distribue ses vélos dans 11 pays."",""pdf""}",null,null,null
"""83f0b8c4-d6be-5789-8a8d-e4a472…",3,5.4137735,"{""presentation_veltis.pdf"",8,""Dates clés"",""Dates clés - 2011 : création de Veltis par Claire Dumont et Hugo Lefèvre. - 2014 : premier Veltis Famille livré. - 2017 : lancement du Veltis Pro L. - 2019 : ouverture du centre logistique de Marnay. - 2022 : lancement du Veltis Trike. - 2026 : présentation du plan stratégique « Cap 2028 ». À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à Valmont des vélos cargo à assistance électrique pour les familles, les professionnels et les collectivités. L'entreprise emploie 182 personnes et distribue ses vélos dans 11 pays."",""pdf""}",null,null,null


## Recherche hybride

Il est possible de combiner les deux recherches mais il faut pour cela une méthode de calcule qui prenne en compte les valeurs renvoyées par nos deux recherches.

La méthode la plus courant utilisée est appelée *Rceiprocal Rank Fusion* (RRF). Le calcul est intuitif. Il s'agit de calculer le rang d'un chunk des deux côtés et d'en faire l'addition.

Un paramètre k est souvent utilisén et permet de contrôler l'effet de l'écart de rang entre les deux vecteurs. Par exemple une valeur élevée renverra un score similaire qu'un article soit premier ou second.

Il est également possible de pondérer les listes de manière à ce qu'une des deux soient plus importantes que l'autre.

In [26]:
def get_rrf_score(dense_rank, sparse_rank, k=0, weights=[1.0, 1.0]):
    score = (1/(dense_rank+k))*weights[0] + (1/(sparse_rank+k))*weights[1]
    return score

print(get_rrf_score(1, 5))
print(get_rrf_score(1, 2))
print(get_rrf_score(1, 3, 60))
print(get_rrf_score(1, 2, 60))

1.2
1.5
0.032266458495966696
0.03252247488101534


In [27]:
def search_hybrid(text, qdrant_client, embed_client, limit=10, search_limit=20, query_filter=None):
    return qdrant_client.query_points(
        COLLECTION,
        prefetch = [
            models.Prefetch(query=embed([text], embed_client)[0], using='dense', limit=search_limit),
            models.Prefetch(query=get_bm25(text), using='sparse', limit=search_limit)
        ],
        query = models.RrfQuery(rrf=models.Rrf(k=60)),
        limit=limit,
        query_filter=query_filter
    ).points

text = "Quel est la stratégie de développement de Veltis dans les prochaines années ?"   
r = search_hybrid(text, qdrant, embedder, 3, 6)
pl.DataFrame(r)

id,version,score,payload,vector,shard_key,order_value
str,i64,f64,struct[5],null,null,null
"""456fb1a3-02e8-515e-a12c-ab3a3b…",3,0.033333,"{""plan_strategique_2026-2028.pptx"",0,"""",""Plan stratégique « Cap 2028 » Comité de direction — janvier 2026 Où en sommes-nous ? 38,4 M€ 9 850 27 % chiffre d'affaires 2025 vélos vendus en 2025 du chiffre d'affaires à l'export Veltis SA — Comité de direction — confidentiel Trois axes stratégiques 1. Professionnels 2. Export 3. Services Devenir la référence du vélo de livraison urbaine auprès des entreprises. Ouvrir un nouveau pays par an en Europe du Nord. Entretien sur site, location longue durée et reconditionnement. Veltis SA — Comité de direction — confidentiel Trajectoire de croissance Chiffre d'affaires annuel, en millions d'euros Veltis SA — Comité de direction — confidentiel Calendrier d'ouverture des marchés export Pays-Bas, Année d'ouverture = 2026. Pays-Bas, Mode d'entrée = Distributeur exclusif. Allemagne, Année d'ouverture = 2027. Allemagne, Mode d'entrée = Filiale commerciale. Danemark, Année d'ouverture = 2028. Danemark, Mode d'entrée = Distributeur exclusif Veltis SA — Comité de direction — confidentiel Investissements clés • Deuxième site de production à Lorvan : 6,5 millions d'euros (sous réserve d'approbation du conseil d'administration) • Atelier de reconditionnement à Marnay : 0,8 million d'euros • Plateforme de suivi connecté des flottes professionnelles : 0,5 million d'euros Veltis SA — Comité de direction — confidentiel Prochaines étapes • Mars 2026 : présentation du projet de deuxième site au conseil d'administration • Avril 2026 : signature du contrat de distribution aux Pays-Bas • Juin 2026 : lancement de l'offre de location longue durée Veltis SA — Comité de direction — confidentiel"",""pptx""}",null,null,null
"""d52a0276-e516-5208-9bbc-7ba48f…",3,0.032787,"{""rapport_annuel_2025.pdf"",11,""7. Perspectives 2026"",""7. Perspectives 2026 Pour accompagner la croissance, la direction soumettra au conseil d'administration un projet de deuxième site de production. Les priorités commerciales de 2026 sont l'ouverture du marché néerlandais et le développement d'une offre de location longue durée. À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à Valmont des vélos cargo à assistance électrique pour les familles, les professionnels et les collectivités. L'entreprise emploie 182 personnes et distribue ses vélos dans 11 pays."",""pdf""}",null,null,null
"""83f0b8c4-d6be-5789-8a8d-e4a472…",3,0.031754,"{""presentation_veltis.pdf"",8,""Dates clés"",""Dates clés - 2011 : création de Veltis par Claire Dumont et Hugo Lefèvre. - 2014 : premier Veltis Famille livré. - 2017 : lancement du Veltis Pro L. - 2019 : ouverture du centre logistique de Marnay. - 2022 : lancement du Veltis Trike. - 2026 : présentation du plan stratégique « Cap 2028 ». À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à Valmont des vélos cargo à assistance électrique pour les familles, les professionnels et les collectivités. L'entreprise emploie 182 personnes et distribue ses vélos dans 11 pays."",""pdf""}",null,null,null
